# 📊 Notebook 01 — Exploratory Data Analysis

Explores the raw Telco Customer Churn dataset to surface patterns and business insights.

**Contents**
1. Load & inspect raw data
2. Missing value analysis
3. Target distribution
4. Churn vs contract type
5. Churn vs monthly charges
6. Churn vs tenure
7. Churn vs internet service
8. Correlation heatmap
9. Summary insights

In [ ]:
import sys, os
sys.path.insert(0, '../src')
sys.path.insert(0, '../visualization')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

from data_preprocessing import load_data, clean_data
from plots import (
    plot_churn_distribution, plot_churn_by_contract,
    plot_churn_by_monthly_charges, plot_churn_by_tenure,
    plot_correlation_heatmap, plot_churn_by_internet_service
)

## 1. Load & Inspect Raw Data

In [ ]:
RAW = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df_raw = load_data(RAW)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include='all').T

## 2. Missing Value Analysis

In [ ]:
# Check raw missing values
print('=== Raw missing values ===')
print(df_raw.isnull().sum())

# TotalCharges has hidden empties (whitespace strings)
tc_empties = (df_raw['TotalCharges'].str.strip() == '').sum()
print(f'\nTotalCharges empty strings: {tc_empties} (customers with tenure=0)')

# Show those rows
df_raw[df_raw['TotalCharges'].str.strip() == ''][['customerID','tenure','MonthlyCharges','TotalCharges']].head(10)

## 3. Target Distribution

In [ ]:
df = clean_data(df_raw)
print(f'Churn rate: {df["Churn"].mean():.2%}')
df['Churn'].value_counts()

In [ ]:
plot_churn_distribution(df)

## 4. Churn vs Contract Type

> **Hypothesis:** Month-to-month customers churn more than long-term contract customers.

In [ ]:
plot_churn_by_contract(df)

# Numeric summary
pd.crosstab(df_raw['Contract'], df_raw['Churn'], normalize='index').mul(100).round(1)

## 5. Churn vs Monthly Charges

> **Hypothesis:** Higher monthly charges correlate with higher churn probability.

In [ ]:
plot_churn_by_monthly_charges(df)

# Statistical test
from scipy import stats
retained = df[df['Churn']==0]['MonthlyCharges']
churned  = df[df['Churn']==1]['MonthlyCharges']
t, p = stats.ttest_ind(retained, churned)
print(f'Retained mean: ${retained.mean():.2f}  |  Churned mean: ${churned.mean():.2f}')
print(f't-statistic: {t:.3f}  |  p-value: {p:.2e}  ({"SIGNIFICANT" if p < 0.05 else "not significant"})') 

## 6. Churn vs Tenure

> **Hypothesis:** Newer customers are more likely to churn.

In [ ]:
plot_churn_by_tenure(df)

# Churn rate by tenure bucket
df_raw2 = df_raw.copy()
df_raw2['TenureBucket'] = pd.cut(df_raw2['tenure'], bins=[0,12,24,36,48,60,72], labels=['0-12','13-24','25-36','37-48','49-60','61-72'], include_lowest=True)
churn_by_tenure = df_raw2.groupby('TenureBucket')['Churn'].apply(lambda x: (x=='Yes').mean()).mul(100).round(1)
print('Churn rate by tenure bucket (%):')
print(churn_by_tenure.to_string())

## 7. Churn vs Internet Service

In [ ]:
plot_churn_by_internet_service(df)

## 8. Correlation Heatmap

In [ ]:
plot_correlation_heatmap(df)

## 9. 💡 Summary Insights

| Finding | Insight |
|---|---|
| Month-to-month contracts | ~3× higher churn than 2-year contracts |
| Monthly charges > $70 | Elevated churn probability |
| Tenure < 12 months | Highest risk segment |
| No TechSupport / OnlineSecurity | Higher churn likelihood |
| Fiber optic internet | Higher churn than DSL (price sensitivity) |

> These patterns should directly guide the feature engineering and model interpretation steps.